# ChromaMark in Jupyter

[ChromaMark](https://github.com/cjfravel-dev/ChromaMark) is a lean rich-markup
format for agent-to-human (and human-to-human) reporting: a strict superset of
Markdown that adds **colored blocks**, **status pills**, **progress meters**,
**key/value fields**, and **collapsible sections** — at a fraction of the token
cost of the equivalent HTML.

Notebooks render HTML + CSS fully, so ChromaMark is a natural fit for capturing
results inline. Install with:

```bash
pip install chromamark
```


In [1]:
from chromamark import ChromaDoc, display_chromamark

## 1 · Render ChromaMark source directly

`display_chromamark(...)` renders a ChromaMark string and displays it inline,
injecting the theme so tones adapt to light/dark automatically.

In [2]:
display_chromamark("""
::: success Model trained
Validation complete — [!ok converged] after 42 epochs.

- best epoch: **37**
- early-stopped on `val_loss`
:::
""")

## 2 · Build a report from your results

`ChromaDoc` is a fluent builder: every method appends a block and returns
`self`, and the object renders inline in Jupyter via `_repr_html_`. The inline
helpers — `.pill()`, `.meter()`, `.tint()` — just return short text tokens you
can drop into any string, heading, table cell, or field value.

In [3]:
# Pretend these came from your analysis / training run.
metrics = {"accuracy": 0.942, "precision": 0.911, "recall": 0.878, "f1": 0.894}
n_holdout = 12_000
dq_warnings = [
    "null_rate(age) = 3.2% (> 1% threshold)",
    "14 duplicate customer_id values",
]
gate_passed = metrics["accuracy"] >= 0.90

In [4]:
cm = ChromaDoc()  # inline helpers (.pill/.meter/.tint) are pure formatters

def status(value, target=0.90):
    return cm.pill("pass") if value >= target else cm.pill("warn", "below target")

acc_pct = f"{metrics['accuracy']:.1%}"
headline = f"Accuracy {cm.pill('ok', acc_pct)} on {n_holdout:,} held-out samples."

doc = (
    ChromaDoc()
    .heading("Model evaluation — churn-v3", level=2)
    .block(
        "success" if gate_passed else "danger",
        "Passed acceptance gate" if gate_passed else "Failed acceptance gate",
        headline,
    )
    .fields({
        "Accuracy":  cm.meter("success", f"{metrics['accuracy']:.1%}"),
        "Precision": cm.meter("info",    f"{metrics['precision']:.1%}"),
        "Recall":    cm.meter("warning", f"{metrics['recall']:.1%}"),
        "F1 score":  cm.meter("info",    f"{metrics['f1']:.1%}"),
    })
    .table(
        ["Metric", "Value", "Status"],
        [
            ["accuracy",  f"{metrics['accuracy']:.3f}",  status(metrics["accuracy"])],
            ["precision", f"{metrics['precision']:.3f}", status(metrics["precision"])],
            ["recall",    f"{metrics['recall']:.3f}",    status(metrics["recall"])],
        ],
    )
    .details(
        f"Data-quality warnings ({len(dq_warnings)})",
        "\n".join(f"- {w}" for w in dq_warnings),
        tone="warning",
    )
)

doc  # renders inline

Metric,Value,Status
accuracy,0.942,PASS
precision,0.911,PASS
recall,0.878,below target


## 3 · It's just text — cheap to emit, easy to diff

The rendered report above compiles from this compact ChromaMark source. It reads
fine as plain text anywhere it isn't rendered, and costs far fewer tokens than
the equivalent HTML — handy when the report is produced by an LLM/agent.

In [5]:
print(doc.to_cm())

## Model evaluation — churn-v3

::: success Passed acceptance gate
Accuracy [!ok 94.2%] on 12,000 held-out samples.
:::

::: fields
Accuracy: [=success 94.2%]
Precision: [=info 91.1%]
Recall: [=warning 87.8%]
F1 score: [=info 89.4%]
:::

| Metric | Value | Status |
| --- | --- | --- |
| accuracy | 0.942 | [!pass] |
| precision | 0.911 | [!pass] |
| recall | 0.878 | [!warn below target] |

::: details warning Data-quality warnings (2)
- null_rate(age) = 3.2% (> 1% threshold)
- 14 duplicate customer_id values
:::



## Notes

- `display_chromamark(src)` renders a string; `ChromaDoc` builds one programmatically
  (`.to_cm()` for the source, `.to_html()` for the fragment).
- Unknown constructs degrade to readable literal text, so nothing breaks in a
  plain Markdown viewer.
- On GitHub's static `.ipynb` preview, output CSS is stripped (colors are lost),
  but structure and text remain — run the notebook in Jupyter/JupyterLab or
  [nbviewer](https://nbviewer.org/) to see the full styling.

Full spec and playground: https://cjfravel-dev.github.io/ChromaMark/
